# Assessment of the reference energy system (2023)

In [1]:
# %pip install brightway2
# %pip install mescal
# %pip install energyscope

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()
LCA_PROJECTS_ROOT = NOTEBOOK_DIR.parent.parent
sys.path.insert(1, str(LCA_PROJECTS_ROOT / '00_Shared'))

In [4]:
from utils import (
    COMMON_DATA_DIR,
    N_capita_2023,
    wood_list, wet_biomass_list, waste_list,
    get_impact_scores,
    default_colors_sankey,
)
from new_plots import _create_sankey_figure, generate_sankey_flows

In [5]:
import os
import pandas as pd
import numpy as np
from energyscope.models import Model
from energyscope.energyscope import Energyscope
from energyscope.result import postprocessing
from ast import literal_eval
import plotly.express as px
import plotly.graph_objects as go
import bw2data as bd
from tqdm import tqdm
from shared.utils import run_model, load_snapshot

In [6]:
import plotly.io as pio
pio.renderers.default = "png"

In [7]:
save_results = False
reference_year = 2023

In [8]:
# AMPL licence 
path_to_ampl_licence = r'C:\Users\matth\ampl' # Path to the AMPL license file
os.environ['PATH'] = path_to_ampl_licence+':'+os.environ['PATH']

In [9]:
LCA_RESULTS_DIR = f'../03_Results/LCA/{reference_year}'

In [10]:
bd.projects.set_current('ecoinvent3.12')

In [11]:
es_tech_df = pd.read_csv(COMMON_DATA_DIR / 'technology_dictionary.csv')

In [12]:
# Create a dict from the Programming Name and Long name columns of es_tech_dict
es_tech_name_dict = dict(zip(es_tech_df['Programming name'], es_tech_df['Long name']))
es_tech_name_dict = {k: v for k, v in es_tech_name_dict.items() if k not in wood_list+wet_biomass_list+waste_list}

## Initialize the model

In [13]:
# Initialize the reference QC model with .mod and .dat files
model = load_snapshot(reference_year)

In [14]:
# Solve the model and get results
results = run_model(model)

Gurobi 12.0.0: 

In [15]:
df_sankey = generate_sankey_flows(
        results=results,
        aggregate_mobility=True,
        aggregate_grid=True,
        aggregate_technology=True,
        run_id=0,
    )
df_sankey['source (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['source']] if x['source'] in es_tech_name_dict else x['source'], axis=1)
df_sankey['target (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['target']] if x['target'] in es_tech_name_dict else x['target'], axis=1)

fig = _create_sankey_figure(df_sankey, colors=default_colors_sankey, long_names=True)

if save_results:
    fig.write_html(f'../03_Results/Figures/reference/sankey_{reference_year}.html')

if save_results:
    df_sankey.to_csv(f'../03_Results/Tables/reference/sankey_raw_{reference_year}.csv', index=False)
df_sankey['value'] *= 1e-3 # from GWh to TWh

## Impact assessment

In [16]:
impact_categories_list = [
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Human health', 'Total human health (biogenic)'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Human health', 'Remaining human health'),
    ('IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total'),
]

In [25]:
scenarios_list = [
    {"model": None, "pathway": None, "year": 2023}, # +1.7°C
    {"model": "image", "pathway": "SSP1-L", "year": 2050}, # +1.7°C
    # {"model": "image", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    # {"model": "image", "pathway": "SSP3-H", "year": 2050}, # +3.6°C
    # {"model": "remind", "pathway": "SSP2-PkBudg1000", "year": 2050}, # +1.8°C
    # {"model": "remind", "pathway": "SSP2-NPi", "year": 2050}, # +2.6°C
    # {"model": "remind", "pathway": "SSP3-rollBack", "year": 2050}, # +3.5°C
    # {"model": "tiam-ucl", "pathway": "SSP2-RCP26", "year": 2050}, # +1.8°C
    # {"model": "tiam-ucl", "pathway": "SSP2-RCP45", "year": 2050}, # +2.8°C
    # {"model": "tiam-ucl", "pathway": "SSP2-Base", "year": 2050}, # +3.1°C
    # {"model": "message", "pathway": "SSP1-L", "year": 2050}, # +1.6°C
    # {"model": "message", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    # {"model": "message", "pathway": "SSP3-H", "year": 2050}, # +3.2°C
]

In [26]:
comparison_2023_2050 = []

for scenario in scenarios_list:

    year = scenario['year']
    model = scenario['model']
    pathway = scenario['pathway']
    if year == 2050:
        path_lca_results = f'../03_Results/LCA/{year}/{model}/{pathway}/'
    else:
        path_lca_results = f'../03_Results/LCA/{year}/'

    # Loading LCA results
    impact_scores = pd.read_csv(path_lca_results+'impact_scores.csv')
    impact_scores_direct = pd.read_csv(path_lca_results+'impact_scores_direct_emissions.csv')

    # from [kg CO2-eq / kW(h) or pkm(/h) or tkm(/h)] to [t-CO2-eq / GW(h) or Mpkm(/h) or Mtkm(/h)]
    impact_scores.Value *= 1e3
    impact_scores_direct.Value *= 1e3

    # Reading impact categories as tuples
    impact_scores.Impact_category = impact_scores.Impact_category.apply(lambda x: literal_eval(x))
    impact_scores_direct.Impact_category = impact_scores_direct.Impact_category.apply(lambda x: literal_eval(x))

    # Merging impact scores with energy configuration results
    df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
        impact_category=impact_categories_list,
        df_impact_scores=impact_scores,
        df_results=results,
    )

    df_annual_prod_direct = get_impact_scores(
        impact_category=impact_categories_list,
        df_impact_scores=impact_scores_direct,
        df_results=results,
        assessment_type='direct',
    )

    for cat in impact_categories_list:

        impact_constr = df_f_mult[cat[-1]].sum()
        impact_op = df_annual_prod[cat[-1]].sum()
        impact_op_direct = df_annual_prod_direct[cat[-1]].sum()
        impact_res_wo_biomass = df_annual_res[~df_annual_res['index'].isin(wood_list+wet_biomass_list+waste_list)][cat[-1]].sum()
        impact_res_biomass = df_annual_res[df_annual_res['index'].isin(wood_list+wet_biomass_list+waste_list)][cat[-1]].sum()

        impact_tot = impact_constr + impact_op + impact_res_wo_biomass + impact_res_biomass
        impact_op_indirect = impact_op - impact_op_direct

        comparison_2023_2050.append([
            cat[-1],
            year,
            model,
            pathway,
            impact_constr,
            impact_op_direct,
            impact_op_indirect,
            impact_res_wo_biomass,
            impact_res_biomass,
            impact_tot,
        ])

comparison_2023_2050 = pd.DataFrame(comparison_2023_2050, columns=['Impact category', 'Year', 'IAM', 'SSP-RCP', 'Construction', 'Operation (direct)', 'Operation (indirect)', 'Resources (wo biomass)', 'Resources (biomass)', 'Total']).set_index(['Impact category', 'Year', 'IAM', 'SSP-RCP'])

In [43]:
adjustment_ratios = []

for scenario in scenarios_list:
    year = scenario['year']
    model = scenario['model']
    pathway = scenario['pathway']

    if year == 2023:
        continue

    for cat in ['Remaining human health', 'Remaining ecosystem quality']:
        tot_2050 = comparison_2023_2050.loc[cat].loc[year].loc[model].loc[pathway]['Total']
        tot_2023 = comparison_2023_2050.loc[cat].loc[2023].loc[np.nan].loc[np.nan]['Total']
        adjustment_ratios.append([cat, model, pathway, tot_2050 / tot_2023])

In [44]:
df_adjustment_ratios = pd.DataFrame(adjustment_ratios, columns=['Impact category', 'IAM', 'SSP-RCP', 'Ratio'])
df_adjustment_ratios.head()

,Impact category,IAM,SSP-RCP,Ratio
0,Remaining human health,image,SSP1-L,0.893574
1,Remaining ecosystem quality,image,SSP1-L,0.927858


In [ ]:
if save_results:
    df_adjustment_ratios.to_csv('../03_Results/Tables/reference/adjustment_ratios.csv', index=False)